# 01 — Load and Clean
**Project:** Retail Sales Intelligence Platform  
**Dataset:** Global Superstore (Kaggle)  
**Purpose:** Load raw CSV, inspect structure, clean column names, fix data types, engineer baseline features, and export a clean CSV for downstream notebooks.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np


In [2]:
# ── Load raw data ─────────────────────────────────────────────────────────
# Relative path from notebooks/ folder — works on any machine
RAW_PATH = os.path.join('..', 'data', 'raw', 'superstore.csv')

df = pd.read_csv(RAW_PATH, encoding='latin-1')
print('Shape:', df.shape)

Shape: (51290, 27)


In [3]:
# ── Inspect: column names ─────────────────────────────────────────────────
print(df.columns.tolist())

['Category', 'City', 'Country', 'Customer.ID', 'Customer.Name', 'Discount', 'Market', 'è®°å½\x95æ\x95°', 'Order.Date', 'Order.ID', 'Order.Priority', 'Product.ID', 'Product.Name', 'Profit', 'Quantity', 'Region', 'Row.ID', 'Sales', 'Segment', 'Ship.Date', 'Ship.Mode', 'Shipping.Cost', 'State', 'Sub.Category', 'Year', 'Market2', 'weeknum']


In [4]:
# ── Inspect: data types ───────────────────────────────────────────────────
print(df.dtypes)

Category              str
City                  str
Country               str
Customer.ID           str
Customer.Name         str
Discount          float64
Market                str
è®°å½æ°           int64
Order.Date            str
Order.ID              str
Order.Priority        str
Product.ID            str
Product.Name          str
Profit            float64
Quantity            int64
Region                str
Row.ID              int64
Sales               int64
Segment               str
Ship.Date             str
Ship.Mode             str
Shipping.Cost     float64
State                 str
Sub.Category          str
Year                int64
Market2               str
weeknum             int64
dtype: object


In [5]:
# ── Inspect: null counts ──────────────────────────────────────────────────
print(df.isnull().sum())

Category          0
City              0
Country           0
Customer.ID       0
Customer.Name     0
Discount          0
Market            0
è®°å½æ°         0
Order.Date        0
Order.ID          0
Order.Priority    0
Product.ID        0
Product.Name      0
Profit            0
Quantity          0
Region            0
Row.ID            0
Sales             0
Segment           0
Ship.Date         0
Ship.Mode         0
Shipping.Cost     0
State             0
Sub.Category      0
Year              0
Market2           0
weeknum           0
dtype: int64


In [6]:
# ── Inspect: first five rows ──────────────────────────────────────────────
df.head(5)

,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,è®°å½æ°,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,111,Consumer,2011-08-09 00:00:00.000,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2011-09-29 00:00:00.000,CA-2011-146969,...,6,Consumer,2011-10-03 00:00:00.000,Standard Class,1.32,California,Paper,2011,North America,40


---
## Cleaning

In [7]:
# ── Step 1: Drop zero-value columns ──────────────────────────────────────
# The column 'è®°å½æ°' (Chinese: 记录数 = record count) is always 1
# for every row and carries no analytical value. Dropping it.
# Market2 groups US+Canada into 'North America' vs Market which keeps them separate.
# We keep Market2 — it is meaningfully different from Market.

GARBAGE_COL = 'è®°å½\x95æ\x95°'
df.drop(columns=[GARBAGE_COL], inplace=True)

print('Columns after drop:', df.shape[1])  # expect 26

Columns after drop: 26


In [8]:
# ── Step 2: Clean column names ────────────────────────────────────────────
# FIX 1: The result must be ASSIGNED back to df.columns
# FIX 2: Columns use '.' as separator, not spaces — replace '.' not ' '
# FIX 3: regex=False prevents '.' being treated as a regex wildcard

df.columns = (
    df.columns
    .str.lower()
    .str.replace('.', '_', regex=False)
    .str.replace(' ', '_', regex=False)
)

print('Renamed columns:', df.columns.tolist())

Renamed columns: ['category', 'city', 'country', 'customer_id', 'customer_name', 'discount', 'market', 'order_date', 'order_id', 'order_priority', 'product_id', 'product_name', 'profit', 'quantity', 'region', 'row_id', 'sales', 'segment', 'ship_date', 'ship_mode', 'shipping_cost', 'state', 'sub_category', 'year', 'market2', 'weeknum']


In [9]:
# ── Step 3: Parse date columns to datetime ────────────────────────────────
# Raw dates are strings: '2011-01-07 00:00:00.000'
# Parsing to datetime64 allows proper date arithmetic downstream

df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date']  = pd.to_datetime(df['ship_date'])

print('order_date dtype:', df['order_date'].dtype)
print('ship_date dtype: ', df['ship_date'].dtype)

order_date dtype: datetime64[us]
ship_date dtype:  datetime64[us]


In [10]:
# ── Step 4: Feature engineering ───────────────────────────────────────────

# 4a. days_to_ship — how long from order to shipment
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days

# 4b. order_month and order_quarter — useful for time-series slicing
df['order_month']   = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter

# Note: 'year' column already exists in raw data (identical to order_date.dt.year)
# We do NOT add a redundant 'order_year' column.

# 4c. profit_margin_pct — profit as % of sales
# FIX: Guard against division by zero (1 row has Sales = 0)
# Result is None/NaN for that row instead of -inf
df['profit_margin_pct'] = np.where(
    df['sales'] != 0,
    (df['profit'] / df['sales'] * 100).round(2),
    np.nan
)

print('New columns added: days_to_ship, order_month, order_quarter, profit_margin_pct')
print('\nSample — zero-sales row (index 46158):')
print(df.loc[46158, ['sales', 'profit', 'profit_margin_pct']])

New columns added: days_to_ship, order_month, order_quarter, profit_margin_pct

Sample — zero-sales row (index 46158):
sales                0.00
profit              -1.11
profit_margin_pct     NaN
Name: 46158, dtype: float64


In [11]:
# ── Step 5: Verify final state ────────────────────────────────────────────

print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

Shape: (51290, 30)

Dtypes:
category                        str
city                            str
country                         str
customer_id                     str
customer_name                   str
discount                    float64
market                          str
order_date           datetime64[us]
order_id                        str
order_priority                  str
product_id                      str
product_name                    str
profit                      float64
quantity                      int64
region                          str
row_id                        int64
sales                         int64
segment                         str
ship_date            datetime64[us]
ship_mode                       str
shipping_cost               float64
state                           str
sub_category                    str
year                          int64
market2                         str
weeknum                       int64
days_to_ship                  int64


In [12]:
# ── Step 6: Export cleaned CSV ────────────────────────────────────────────
# index=False prevents pandas from saving the row index as 'Unnamed: 0'

OUTPUT_PATH = os.path.join('..', 'data', 'processed', 'cleaned_superstore.csv')
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)
print('Saved cleaned CSV to:', OUTPUT_PATH)
print('Final shape:', df.shape)

Saved cleaned CSV to: ..\data\processed\cleaned_superstore.csv
Final shape: (51290, 30)


## Loading Data Into MySQL

In [ ]:
# ── Database Connection ────────────────────────────────────────────

from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path('..')/'.env')


username = os.environ.get("MYSQL_USER", "root")
password = os.environ.get("MYSQL_PASSWORD")
host = "localhost"
database = "retail_sales"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

connection = engine.connect()

print("Connection Successful!")

Connection Successful!


In [15]:
# ── Load data into MySQL ──────────────────────────────────────────────────
df.to_sql(
    name='superstore',
    con=engine,
    if_exists='replace',
    index=False
)

print("Data loaded successfully!")

Data loaded successfully!


In [17]:
# ── Connection closed ────────────────────────────────────────────
connection.close()
engine.dispose()

print("Database connection closed.")

Database connection closed.
